In [1]:
import json
import re
import json
import string
from nltk import ngrams
from spello.model import SpellCorrectionModel
import pandas as pd
import copy

In [2]:
survey_data = pd.read_excel("./NLP Structure Survey Results (2023-05-31).xlsx")
survey_data

,DATE ADDED,Survey\nQuery\nRespondent,Survey\nQuestion,Response,Category order,Certification - 1,Feature - 2,Application - 3,Property - 4,Units - 5,Polymer - 6,Filler type - 7,Filler % - 8,Brand - 9,Competitor grade - 10
0,2023-05-31,S01-R06,S01-Q01,Looking for a material with a tensile modulus ...,4,0,0,0,Tensile modulus of 3000 Mpa,0,0,0,0,0,0
1,2023-05-31,S01-R06,S01-Q02,PA Tm>=300C,"6,4",0,0,0,Melt temp of at least 300°C,0,Nylon,0,0,0,0
2,2023-05-31,S01-R06,S01-Q03,NSF61 certification at 23C,1,NSF61 certification at 23°C,0,0,0,0,0,0,0,0,0
3,2023-05-31,S01-R06,S01-Q04,"unfilled, V0, density <= XXX(the density speci...","7,2(2),4,3",0,"Light weighting, Thermally conductive",Will be used as a component in a light aligner,Flame rating of V0,0,0,Unfilled,0,0,0
4,2023-05-31,S01-R06,S01-Q05,"PPS, and hydrolysis resistant","6,3",0,0,Part of a water pump-impeller,0,0,PPS,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2798,NaT,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0
2799,NaT,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0
2800,NaT,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0
2801,NaT,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0


In [3]:
survey_data['Response'] = survey_data['Response'].apply(lambda x: str(x).strip())
survey_data['Response'] = survey_data['Response'].apply(lambda x: str(x).strip("\n"))
survey_data['Response'] = survey_data['Response'].apply(lambda x: ", ".join(str(x).split("\n")))

In [4]:
queries_uncleaned = list(set(survey_data["Response"].to_list()))

## Note: Pre-processing the query is optional. If you dont want skip to final step

## update the below code using the code in NER Pipeline only if you want to include cleaned queries (original + cleaned)

In [5]:
SPELL_CORRECTOR = SpellCorrectionModel(language='en')
SPELL_CORRECTOR.load(
    model_path='./cleaning_dependencies/spell_correction_model/model.pkl')

In [6]:
def get_german_mapping():
    with open(file="./cleaning_dependencies/german_terms_mapping.json") as f_in:
        german_mapping = json.load(fp=f_in)
    return german_mapping

In [7]:
def check_spelling(text) -> str:
    # Tokenize the text
    words = text.split()

    corrected = [SPELL_CORRECTOR.spell_correct(
        text=word)['spell_corrected_text'] if (word.isalpha() and len(str(word))>3) else word for word in words]
    return " ".join(corrected)

In [8]:
def clean_cti_property(q) -> str:
    keywords = ['cti', 'comparative tracking index', 'comparative tracking index rating', 'comparative tracking rating',
                'comparative tracking']

    for keyword in keywords:
        if keyword in q:
            q = q.replace(f"{keyword} plc", f"{keyword} plc ")

            q = q.replace(f"{keyword} = plc", f"{keyword} plc ")
            q = q.replace(f"{keyword} of plc =", f"{keyword} plc ")
            q = q.replace(f"{keyword} of plc", f"{keyword} plc ")
            q = q.replace(f"{keyword} is plc", f"{keyword} plc ")
            q = q.replace(f"{keyword} as plc", f"{keyword} plc ")
            q = q.replace(f"{keyword} with plc =", f"{keyword} plc ")
            q = q.replace(f"{keyword} with plc", f"{keyword} plc ")
            q = q.replace(f"{keyword} select plc =", f"{keyword} plc ")
            q = q.replace(f"{keyword} select plc", f"{keyword} plc ")

            # for num in range(0, 7):
            #     q = q.replace(f"{keyword} {num} ", f"{keyword} plc {num} ")
            #     q = q.replace(f"{keyword} = {num} ", f"{keyword} plc {num} ")
            #     q = q.replace(f"{keyword} of {num} ", f"{keyword} plc {num} ")
            #     q = q.replace(f"{keyword} is {num} ", f"{keyword} plc {num} ")
            #     q = q.replace(f"{keyword} as {num} ", f"{keyword} plc {num} ")
            #     q = q.replace(f"{keyword} with {num} ", f"{keyword} plc {num} ")
            #     q = q.replace(f"{keyword} select {num} ", f"{keyword} plc {num} ")

            # max_value = 6
            # pattern = rf'\b{keyword}\s+(\d+)\b'
            # q = re.sub(pattern, lambda match: f"{keyword} plc {match.group(1)}" if int(match.group(1)) <= max_value and not match.group(1).isalpha() else match.group(0), q)

            # pattern = rf'\b{keyword}\s+=\s+(\d+)\b'
            # q = re.sub(pattern, lambda match: f"{keyword} plc {match.group(1)}" if int(match.group(1)) <= max_value and not match.group(1).isalpha() else match.group(0), q)

            # pattern = rf'\b{keyword}\s+of\s+(\d+)\b'
            # q = re.sub(pattern, lambda match: f"{keyword} plc {match.group(1)}" if int(match.group(1)) <= max_value and not match.group(1).isalpha() else match.group(0), q)

            # pattern = rf'\b{keyword}\s+is\s+(\d+)\b'
            # q = re.sub(pattern, lambda match: f"{keyword} plc {match.group(1)}" if int(match.group(1)) <= max_value and not match.group(1).isalpha() else match.group(0), q)

            # pattern = rf'\b{keyword}\s+as\s+(\d+)\b'
            # q = re.sub(pattern, lambda match: f"{keyword} plc {match.group(1)}" if int(match.group(1)) <= max_value and not match.group(1).isalpha() else match.group(0), q)

            # pattern = rf'\b{keyword}\s+with\s+(\d+)\b'
            # q = re.sub(pattern, lambda match: f"{keyword} plc {match.group(1)}" if int(match.group(1)) <= max_value and not match.group(1).isalpha() else match.group(0), q)

            # pattern = rf'\b{keyword}\s+select\s+(\d+)\b'
            # q = re.sub(pattern, lambda match: f"{keyword} plc {match.group(1)}" if int(match.group(1)) <= max_value and not match.group(1).isalpha() else match.group(0), q)

    # remove extra spaces
    q = re.sub("(\s+)", " ", q.strip())
    return q

In [9]:
def get_filtered_values() -> list:
    filtered_values = []
    units_list = []
    for k in UNIQUE_VALUES:
        for value in UNIQUE_VALUES[k]:
            words = value.split()
            for word in words:
                if (re.search("\.", word) or re.search("-", word) or re.search("/", word)) and (word not in ["-", "/", "."]):
                    filtered_values.append(word)
                    if k == "UNIT":
                        units_list.append(word)
    units_list += ["g/ml", "w/m.k", "kg/m^3", "g/cm^3", "kj/m^2", "g/cm^3", "grams/cm^2", "g/cc", "ohm/m",
                   "kj/m^3", "g/cc", "kg/m3", "kn/m", "kv/mm", "kg/m3", "g/cm3", "j/m^2", "kg/cc", "kn/m", "w/m.k"]
    filtered_values = list(set(filtered_values + units_list))
    return filtered_values, units_list

In [10]:
def add_spaces(text, values, units) -> str:
    words = text.split()
    updated_words = []
    for word in words:
        #         if any(word.startswith(v) for v in values):
        if any(word in v for v in values) or any(word[1:] in v for v in values) or any(word[:-1] in v for v in values) or \
                any(word.endswith(v) for v in units) or any(word[:-1].endswith(v) for v in units):
            # Word is substring in one of the lists, add word
            #             print(word)
            updated_words.append(word)
        else:
            processed_word = word.replace("/", " / ")  # .replace("-", " -")
            pattern = r"(?<=\D)-(?=\d)|(?<=\d)-(?=\D)|(?<=\D)-(?=\D)"
            processed_word = re.sub(pattern, ' - ', processed_word)
            # 9.0x10^ -6 >>>>>>>>>> 9.0x10^-6
            processed_word = re.sub(
                r"(?<=\d)\^ - (?=\d)", '^-', processed_word)
            # 9.0e -6 >>>>>>>> 9.0e-6
            processed_word = re.sub(r"(?<=\d)e - (?=\d)", 'e-', processed_word)
            dot_pattern = r'(?<=[a-zA-Z])\.(?=[a-zA-Z])|(?<=[a-zA-Z])\.(?=\d)|(?<=\d)\.(?=[a-zA-Z])'
            processed_word = re.sub(dot_pattern, '. ', processed_word)
            updated_words.append(processed_word)

    cleaned_text = " ".join(updated_words)
    cleaned_text = re.sub("(\s+)", " ", cleaned_text.strip())
    cleaned_text = cleaned_text.replace("+/ -", "+/-")
    return cleaned_text


# Trim all the enities
def remove_brackets(labelled_text) -> str:
    for i in [("(", ")"), ("[", "]"), ("{", "}")]:
        if i[0] in labelled_text and not i[1] in labelled_text:
            labelled_text = labelled_text.replace(i[0], " ")
        elif i[1] in labelled_text and not i[0] in labelled_text:
            labelled_text = labelled_text.replace(i[1], " ")

#     # <> not include
#     labelled_text = labelled_text.strip("()").strip(
#         "[]").strip("{}").strip('""').strip("''").strip()
    if labelled_text:
        for i in [("(", ")"), ("[", "]"), ("{", "}"), ("'", "'"), ('"', '"')]:
            if i[0] == labelled_text[0] and i[1] == labelled_text[-1]:
                labelled_text = labelled_text[1:-1]
                break

    return labelled_text


def remove_unwanted_characters(labelled_text) -> str:
    trim_characters = ["=", "/", "-", ",", ":",
                       ";", "#", "$", "&", "*", "?", "|", "@"]
    if labelled_text and len(labelled_text) > 1 and labelled_text[0] in trim_characters and not labelled_text[1:].split()[0][0].isnumeric():  # or
        labelled_text = labelled_text[1:]

    if labelled_text and labelled_text[-1] in trim_characters:
        labelled_text = labelled_text[:-1]
    return labelled_text.strip()


def remove_hyphens(labelled_text) -> str:
    # converts "- -" to "-"
    labelled_text = re.sub(r'(- ){2,}', r'-', labelled_text)
    pattern = r'^(-+)(?![-.\d])|(?<![-.\d])(-+)$'
    labelled_text = re.sub(pattern, '', labelled_text)
    labelled_text = re.sub("(\s+)", " ", labelled_text.strip())
    return labelled_text


def trim(labelled_text) -> str:
    labelled_text_len = len(labelled_text)
    cleaned_labelled_text = remove_brackets(labelled_text)

    cleaned_labelled_text = remove_hyphens(cleaned_labelled_text)

    cleaned_labelled_text = remove_unwanted_characters(cleaned_labelled_text)

    if len(cleaned_labelled_text) == labelled_text_len:
        return cleaned_labelled_text
    else:
        return trim(cleaned_labelled_text)

In [11]:
GERMAN_MAPPING = get_german_mapping()

In [12]:
def convert_property_shortforms_with_values(labelled_text) -> str:
    pattern = r"\d+\.\d+sg|sg\d+\.\d+|\d+sg|sg\d+|\d+\.\d+hdt|hdt\d+\.\d+|\d+hdt|hdt\d+"
    matches = re.findall(pattern, labelled_text)
    for match in matches:
        numbers = re.findall(r'[0-9.]+', match)
        text = re.findall(r'[a-z]+', match)
        labelled_text = labelled_text.replace(match, f"{text[0]} of {numbers[0]}", 1)
    return  labelled_text

In [13]:
def data_preprocessing(text, values, units) -> str:
    query = copy.deepcopy(text)

    # remove $, #, :
    query = query.replace('$', ' ').replace('#', ' ').replace(":", ' ')

    # replace 
    query = query.replace(";", ", ").replace("；", ", ").replace("~", '- ').replace("–", "-")
    

    # remove extra spaces
    query = re.sub("(\s+)", " ", query.strip())

    # trim
    query = trim(query)

    # replace multiple occurance with single occurance
    query = re.sub(r'[!@%?\'=&\.]+', lambda match: match.group(0)[0], query)
    query = re.sub(r'[,]+', lambda match: match.group(0)[0], query)

    # replace special characters to right format
    query = query.replace('（', "(").replace('）', ')').replace('ºc', "°c").replace(
        '˚c', "°c").replace('℃', "°c").replace('=<', "<=").replace('=>', '>=').lower()

    # find all occurrences of the pattern and add a space after each match
    pattern1 = r'(?<=[<>=!])(?=[^=])|(?<=[^<>=])(?=[<>=!])'
    query = re.sub(pattern1, ' ', query).replace("! =", "!=")

    # add space before and after for "+" "@" "|" "&"
    # we can include more characters inside the [] like r'([+,/-])'
    query = re.sub(r'([:&+@|()\[\]\{\}])', r' \1 ', query)
    # [{]}
    # add space after for ";", ":" "?" "%" ")"
    query = re.sub(r'([;?%])', r'\1 ', query)

    # add space before for "("
#     query = re.sub(r'([(]', r' \1', query)

    # add space after for ","
    pattern2 = r"(?<=\D),(?=\d)|(?<=\d),(?=\D)|(?<=\D),(?=\D)"
    query = re.sub(pattern2, ', ', query)

    # after adding spaces, we are removing if any multiple spaces exist
    query = re.sub("(\s+)", " ", query.strip())

#     .replace("( ", "(").replace(" )", ")").replace("[ ", "[").replace(" ]", "]").replace("? ? ?", "???").replace("? ?", "??")
    # remove unnecessary spaces
    query = query.replace("+ /", "+/")

    # converting some cases to right format
    # replace("<= >=", "<==>").
    query = query.replace("> / =", ">=").replace("< / =", "<=").replace(
        "= / >", ">=").replace("= / <", "<=").replace("< >=", "<=>")

    # add space for ".", "/", "-"
    query = add_spaces(query, values, units)

    #convert sg1.4, hdt200 to proper format
    query = convert_property_shortforms_with_values(query)

    query = clean_cti_property(query)

    possible_terms_for_german_detection = []
    search_query_splitted = query.split()
    search_query_splitted = [word.strip(
        string.punctuation) for word in search_query_splitted]
    threegrams = ngrams(search_query_splitted, 3)
    twograms = ngrams(search_query_splitted, 2)
    onegrams = ngrams(search_query_splitted, 1)

    # GERMAN_MAPPING = get_german_mapping()
    possible_terms_for_german_detection = [
        " ".join(grams) for grams in threegrams if " ".join(grams) in GERMAN_MAPPING]
    possible_terms_for_german_detection.extend(
        [" ".join(grams) for grams in twograms if " ".join(grams) in GERMAN_MAPPING])
    possible_terms_for_german_detection.extend(
        [" ".join(grams) for grams in onegrams if " ".join(grams) in GERMAN_MAPPING])

    for german_term in possible_terms_for_german_detection:
        query = query.replace(german_term, GERMAN_MAPPING[german_term])

    #query = check_spelling(query)

    query = query.replace("performance level category", "plc")

    # remove extra spaces (do always just before return)
    query = re.sub("(\s+)", " ", query.strip())
    return query

In [14]:
with open('./unique_values_31_08_23.json', 'r') as fp:
    UNIQUE_VALUES = json.load(fp)

In [15]:
filtered_values, units_list = get_filtered_values()

In [16]:
queries_cleaned = []
for q in queries_uncleaned:
    q_cleaned = data_preprocessing(q, filtered_values, units_list)
    queries_cleaned.append(q_cleaned)

# queries_cleaned

## Final Step

In [19]:
print(len(queries_uncleaned))
print(len(queries_uncleaned + queries_cleaned))
print(len(list(set(queries_uncleaned + queries_cleaned))))

1530
3060
3016


In [20]:
pre_process = True

In [21]:
if pre_process:
    queries_list_prodigy = [{'text': q} for q in list(set(queries_uncleaned + queries_cleaned))]
else:
    queries_list_prodigy = [{'text': q} for q in queries_uncleaned]

In [22]:
with open("./input/NLP_Structure_Survey_Results_31_05_23_all.jsonl", "w", encoding="utf-8") as jsonfile:
    jsonfile.write("\n".join([json.dumps(i) for i in queries_list_prodigy]))

## Divide the queries into batches

In [23]:
import numpy as np
import random


random.shuffle(queries_list_prodigy)

batches = np.array_split(queries_list_prodigy, 3)
# batches

In [24]:
for batch_no, batch in enumerate(batches):
    with open(f"./input/NLP_Structure_Survey_Results_31_05_23_{batch_no+1}.jsonl", "w", encoding="utf-8") as jsonfile:
        jsonfile.write("\n".join([json.dumps(i) for i in batch.tolist()]))